# 4. Six records you can check by hand

**Estimated session:** 25–40 minutes; allow time to explain your reasoning aloud.

| Before running | Your task |
|---|---|
| Input | `data/practice_farms.csv`: six explicitly invented records |
| Working file | Open `scripts/04-small-farm-example.R`; Save As `scripts/my-04-small-farm-example.R` |
| Starting state | Open the project and restart R; no previous objects are needed |
| Stop after | The `toy-save` block |
| Finish with | `outputs/toy/error-table.csv` and a hand calculation |

## 4.1 Read one row before running a model

Every number on this page is invented. One row is one assay result from a farm at a sampling round. The unit is CFU/mL. Here we use the original concentration scale so that we can calculate everything by hand.

| Farm | Sampling round | Invented count |
|---|---|---:|
| A | 1 | 10 |
| A | 2 | 10 |
| B | 1 | 30 |
| B | 2 | 30 |
| C | 1 | 10 |
| C | 2 | 30 |

A1 and A2 come from the same farm at different rounds. They are related observations, even though they occupy separate rows. This is familiar from repeatedly measuring the same biological source.

In [1]:
toy <- read.csv("data/practice_farms.csv", stringsAsFactors = FALSE)
toy
stopifnot(nrow(toy) == 6, length(unique(toy$FarmID)) == 3)

FarmID,Sampling,count_cfu_ml
<chr>,<int>,<int>
A,1,10
A,2,10
B,1,30
B,2,30
C,1,10
C,2,30


`unique()` returns each distinct label once. We have six records and three farms.

## 4.2 Decide which farm is held out

Before calculating error, we choose C as the test farm. Both C records must stay together. A and B supply training data. This manual choice is for illustration; Chapter 6 will use a fixed random selection of whole farms.

`==` asks “equal to”; `!=` asks “not equal to”. Inside `[rows, columns]`, the empty position after the comma retains all columns.

In [2]:
# Decide to hold out C before calculating a score.
toy_train <- toy[toy$FarmID != "C", ]
toy_test <- toy[toy$FarmID == "C", ]
data.frame(FarmID = c("A", "B", "C"), role = c("train", "train", "test"))

FarmID,role
<chr>,<chr>
A,train
B,train
C,test


`intersect()` lists shared labels. `character(0)` means no shared labels:

In [3]:
intersect(unique(toy_train$FarmID), unique(toy_test$FarmID))
stopifnot(nrow(toy_train) == 4, nrow(toy_test) == 2)

character(0)

**Pause:** point to the two test rows in the table before proceeding. We will not use their outcomes to compute the training average.

## 4.3 Why start with one average?

Training counts are 10, 10, 30 and 30. Their sum is 80 across four records, so the training mean is **20 CFU/mL**. Without sample-specific information, we can give both new samples this same transparent prediction. This is a fitted rule with one learned value: a baseline.

`rep(value, times)` repeats a value. We need one prediction for each of the two test rows:

In [4]:
toy_mean <- mean(toy_train$count_cfu_ml)
toy_mean
toy_test$prediction <- rep(toy_mean, nrow(toy_test))

[1] 20

A later model may use more information to improve on this starting point. Complexity alone does not establish an improvement.

## 4.4 Calculate an error before reading its formula

The test measurements are 10 and 30. Subtract the prediction, 20, from each one:

| Observed | Predicted | Residual: observed − predicted | Absolute error | Squared error |
|---:|---:|---:|---:|---:|
| 10 | 20 | −10 | 10 | 100 |
| 30 | 20 | 10 | 10 | 100 |

`abs()` removes the sign; `^2` squares a number. A negative residual means the prediction was above the observed value. It does not mean there was a negative microbial count.

In [5]:
toy_test$error <- toy_test$count_cfu_ml - toy_test$prediction
toy_errors <- data.frame(observed = toy_test$count_cfu_ml, predicted = toy_test$prediction,
                         residual = toy_test$error, absolute_error = abs(toy_test$error),
                         squared_error = toy_test$error^2)
toy_errors

observed,predicted,residual,absolute_error,squared_error
<int>,<dbl>,<dbl>,<dbl>,<dbl>
10,20,-10,10,100
30,20,10,10,100


- **MAE:** average the absolute errors: `(10 + 10) / 2 = 10`.
- **RMSE:** average the squared errors, then take the square root: `sqrt((100 + 100) / 2) = 10`. `sqrt()` is the square-root function.

In [6]:
toy_rmse <- sqrt(mean(toy_test$error^2))
toy_mae <- mean(abs(toy_test$error))
data.frame(RMSE_CFU_ml = toy_rmse, MAE_CFU_ml = toy_mae)

RMSE_CFU_ml,MAE_CFU_ml
<dbl>,<dbl>
10,10


Both are **10 CFU/mL** in this deliberately symmetric example. They need not be equal for other errors. Neither is an accuracy percentage.

```{admonition} Open after you try: what if the two residuals were −10 and 30?
:class: tip dropdown
MAE would be (10 + 30) / 2 = 20. RMSE would be sqrt((100 + 900) / 2), about 22.36. Squaring makes a larger error contribute more strongly. These are hand-calculation alternatives, not replacement results for our saved six-row table.
```

## 4.5 A split that runs but answers a different question

The following counterexample assigns alternate rows to two sets. `c(1, 3, 5)` supplies row positions. It uses separate `bad_` objects, leaving our correct toy split unchanged:

In [7]:
# A separate counterexample; it does not replace toy_train or toy_test.
bad_train <- toy[c(1, 3, 5), ]
bad_test <- toy[c(2, 4, 6), ]
intersect(unique(bad_train$FarmID), unique(bad_test$FarmID))

[1] "A" "B" "C"

The overlap contains **A, B and C**. Both sets contain records from every farm. This cannot evaluate prediction for an unseen farm. The code ran; the split did not match the stated prediction setting.

**Ask AI:** paste this six-row table and ask it to explain the difference between the two splits. Require it to list the shared farm labels. Verify its list yourself before accepting the explanation.

## 4.6 Save and explain

In [8]:
dir.create("outputs/toy", recursive = TRUE, showWarnings = FALSE)
write.csv(toy_errors, "outputs/toy/error-table.csv", row.names = FALSE)
stopifnot(toy_mean == 20, toy_rmse == 10, toy_mae == 10)
stopifnot(length(intersect(toy_train$FarmID, toy_test$FarmID)) == 0)
cat("Toy checks passed: mean 20, MAE 10, RMSE 10; all counts are invented.\n")

Toy checks passed: mean 20, MAE 10, RMSE 10; all counts are invented.


Open **Files → outputs → toy → error-table.csv**. Check the two residuals against the hand table. Record the file in your completion card.

**Checkpoint 4:** explain why C has two test records, why the prediction is 20, and why the error metrics here have CFU/mL units. The real example will use a log10 target, so its numerical errors cannot be directly compared with these original-scale toy errors.

**Next:** [meet the real milk data](05-meet-real-data.ipynb).